In [524]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import warnings
warnings.filterwarnings('ignore') # supress the warnings

import os
import sqlalchemy
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

user = os.getenv("MYSQL_USER")
password = os.getenv("MYSQL_PASSWORD")
host = os.getenv("MYSQL_HOST")
database = os.getenv("MYSQL_DATABASE")

engine = sqlalchemy.create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:3306/{database}"
)

conn = engine.connect()

pd.read_sql("SHOW TABLES;", conn)

,Tables_in_ca2_agriculture


# Datasets Inspection & Ingestion 

## Milk Production Dataset

First, lets analyze the total production of milk available on the farm of each EU country.

In [525]:
df_milk_prod =  pd.read_csv("data/raw/milk_production.csv")
df_milk_prod.head()

,DATAFLOW,LAST UPDATE,freq,dairyprod,milkitem,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:APRO_MK_FARM$DEFAULTVIEW(1.0),04/12/25 11:00:00,Annual,Farm milk products delivered to dairies,Utilization of whole milk (1 000 t),Austria,2015,3120.58,NaN,NaN
1,ESTAT:APRO_MK_FARM$DEFAULTVIEW(1.0),04/12/25 11:00:00,Annual,Farm milk products delivered to dairies,Utilization of whole milk (1 000 t),Austria,2016,3114.43,NaN,NaN
2,ESTAT:APRO_MK_FARM$DEFAULTVIEW(1.0),04/12/25 11:00:00,Annual,Farm milk products delivered to dairies,Utilization of whole milk (1 000 t),Austria,2017,3208.80,NaN,NaN
3,ESTAT:APRO_MK_FARM$DEFAULTVIEW(1.0),04/12/25 11:00:00,Annual,Farm milk products delivered to dairies,Utilization of whole milk (1 000 t),Austria,2018,3203.83,NaN,NaN
4,ESTAT:APRO_MK_FARM$DEFAULTVIEW(1.0),04/12/25 11:00:00,Annual,Farm milk products delivered to dairies,Utilization of whole milk (1 000 t),Austria,2019,3165.87,NaN,NaN


In [526]:
df_milk_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1939 entries, 0 to 1938
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     1939 non-null   object 
 1   LAST UPDATE  1939 non-null   object 
 2   freq         1939 non-null   object 
 3   dairyprod    1939 non-null   object 
 4   milkitem     1939 non-null   object 
 5   geo          1939 non-null   object 
 6   TIME_PERIOD  1939 non-null   int64  
 7   OBS_VALUE    1913 non-null   float64
 8   OBS_FLAG     182 non-null    object 
 9   CONF_STATUS  26 non-null     object 
dtypes: float64(1), int64(1), object(8)
memory usage: 151.6+ KB


In [527]:
df_milk_prod.describe(include='object')

,DATAFLOW,LAST UPDATE,freq,dairyprod,milkitem,geo,OBS_FLAG,CONF_STATUS
count,1939,1939,1939,1939,1939,1939,182,26
unique,1,1,1,5,2,37,5,1
top,ESTAT:APRO_MK_FARM$DEFAULTVIEW(1.0),04/12/25 11:00:00,Annual,"Raw milk, total available on farms",Utilization of whole milk (1 000 t),Austria,p,C
freq,1939,1939,1939,660,1301,60,133,26


In [528]:
df_milk_prod["dairyprod"].unique()

array(['Farm milk products delivered to dairies',
       'Milk products, other than drinking milk, cream, butter and cheese, delivered to dairies',
       'Milk products, other than milk and cream, delivered to dairies',
       'Raw milk, total available on farms',
       'Raw milk delivered to dairies'], dtype=object)

In [529]:
df_milk_prod.milkitem.unique()

array(['Utilization of whole milk (1 000 t)',
       'Products obtained (1 000 t)'], dtype=object)

In [530]:
milk_prod = df_milk_prod[df_milk_prod["milkitem"] == "Products obtained (1 000 t)"][["geo","TIME_PERIOD","dairyprod","OBS_VALUE"]].reset_index(drop=True)
milk_prod.head()

,geo,TIME_PERIOD,dairyprod,OBS_VALUE
0,Austria,2015,"Milk products, other than milk and cream, deli...",0.0
1,Austria,2016,"Milk products, other than milk and cream, deli...",0.0
2,Austria,2017,"Milk products, other than milk and cream, deli...",0.0
3,Austria,2018,"Milk products, other than milk and cream, deli...",0.0
4,Austria,2019,"Milk products, other than milk and cream, deli...",0.0


In [531]:
milk_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 638 entries, 0 to 637
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   geo          638 non-null    object 
 1   TIME_PERIOD  638 non-null    int64  
 2   dairyprod    638 non-null    object 
 3   OBS_VALUE    630 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 20.1+ KB


Let's pivot the dataframe to convert the required features from long format to wide format, with only the observed relevant variables.

In [532]:
milk_prod= milk_prod.pivot_table(index=["geo","TIME_PERIOD"],columns="dairyprod",values="OBS_VALUE").reset_index()
milk_prod.columns.name = None  # Resetting column index name
milk_prod

,geo,TIME_PERIOD,"Milk products, other than milk and cream, delivered to dairies","Raw milk, total available on farms"
0,Austria,2015,0.0,3568.90
1,Austria,2016,0.0,3659.96
2,Austria,2017,0.0,3747.78
3,Austria,2018,0.0,3859.99
4,Austria,2019,0.0,3820.04
...,...,...,...,...
323,United Kingdom,2015,0.0,15457.00
324,United Kingdom,2016,0.0,14938.00
325,United Kingdom,2017,NaN,15443.00
326,United Kingdom,2018,0.0,15488.11


In [533]:
milk_prod.describe()

,TIME_PERIOD,"Milk products, other than milk and cream, delivered to dairies","Raw milk, total available on farms"
count,328.000000,302.000000,328.000000
mean,2019.435976,2.507450,12941.395884
std,2.850321,40.726065,33666.835111
min,2015.000000,0.000000,0.000000
25%,2017.000000,0.000000,916.970000
50%,2019.000000,0.000000,2406.140000
75%,2022.000000,0.000000,8570.272500
max,2024.000000,707.500000,174013.610000


From the previous outputs, the feature "Milk products, other than milk and cream, delivered to dairies" seems to contain only nan or 0 values, lets analyze it. Lets see if it contains non null values for Ireland , in that case it can be kept for other countries as well.

In [534]:
((milk_prod["geo"] == "Ireland") & (milk_prod["Milk products, other than milk and cream, delivered to dairies"].fillna(0) != 0)).any()

np.False_

Since the output is false, the feature "Milk products, other than milk and cream, delivered to dairies" can be dropped, while the "Raw milk, total available on farms" can be renamed for better readability.

In [535]:
milk_prod = milk_prod.drop(columns=["Milk products, other than milk and cream, delivered to dairies"])
milk_prod = milk_prod.rename(columns={
    "geo" : "country",
    "TIME_PERIOD": "year",
    "Raw milk, total available on farms": "milk_prod_1000t"
})
milk_prod.head()

,country,year,milk_prod_1000t
0,Austria,2015,3568.90
1,Austria,2016,3659.96
2,Austria,2017,3747.78
3,Austria,2018,3859.99
4,Austria,2019,3820.04


In [536]:
# Lets store the cleaned data table to the database.
milk_prod.to_csv("data/processed/milk_prod.csv",index=False)

## Milk Delivery and Consumption Dataset

In [537]:
df_products_milk =  pd.read_csv("data/raw/milk_delivery_&_consumption.csv")
df_products_milk.head()

,DATAFLOW,LAST UPDATE,freq,milkitem,dairyprod,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:APRO_MK_COLA$DEFAULTVIEW(1.0),13/05/26 23:00:00,Annual,Fat content (t),Raw cows' milk delivered to dairies,Percentage,Cyprus,2024,4.07,NaN,NaN
1,ESTAT:APRO_MK_COLA$DEFAULTVIEW(1.0),13/05/26 23:00:00,Annual,Fat content (t),Raw cows' milk delivered to dairies,Percentage,France,2024,4.11,p,NaN
2,ESTAT:APRO_MK_COLA$DEFAULTVIEW(1.0),13/05/26 23:00:00,Annual,Product obtained,Raw cows' milk delivered to dairies,Thousand tonnes,Albania,2017,57.36,p,NaN
3,ESTAT:APRO_MK_COLA$DEFAULTVIEW(1.0),13/05/26 23:00:00,Annual,Product obtained,Raw cows' milk delivered to dairies,Thousand tonnes,Albania,2018,64.39,NaN,NaN
4,ESTAT:APRO_MK_COLA$DEFAULTVIEW(1.0),13/05/26 23:00:00,Annual,Product obtained,Raw cows' milk delivered to dairies,Thousand tonnes,Albania,2019,56.82,NaN,NaN


In [538]:
df_products_milk.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1725 entries, 0 to 1724
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     1725 non-null   object 
 1   LAST UPDATE  1725 non-null   object 
 2   freq         1725 non-null   object 
 3   milkitem     1725 non-null   object 
 4   dairyprod    1725 non-null   object 
 5   unit         1725 non-null   object 
 6   geo          1725 non-null   object 
 7   TIME_PERIOD  1725 non-null   int64  
 8   OBS_VALUE    1488 non-null   float64
 9   OBS_FLAG     222 non-null    object 
 10  CONF_STATUS  237 non-null    object 
dtypes: float64(1), int64(1), object(9)
memory usage: 148.4+ KB


In [539]:
df_products_milk.unit.unique()

array(['Percentage', 'Thousand tonnes'], dtype=object)

In [540]:
df_products_milk.dairyprod.unique()

array(["Raw cows' milk delivered to dairies",
       'Raw cream delivered to dairies (in milk equivalent)',
       'Drinking milk', 'Cream for direct consumption',
       'Milk and cream powders, excluding skimmed milk powders'],
      dtype=object)

In [541]:
df_products_milk = df_products_milk[df_products_milk["unit"] == "Thousand tonnes"][["geo","TIME_PERIOD","dairyprod","OBS_VALUE"]].reset_index(drop=True)
df_products_milk.head()

,geo,TIME_PERIOD,dairyprod,OBS_VALUE
0,Albania,2017,Raw cows' milk delivered to dairies,57.36
1,Albania,2018,Raw cows' milk delivered to dairies,64.39
2,Albania,2019,Raw cows' milk delivered to dairies,56.82
3,Albania,2020,Raw cows' milk delivered to dairies,56.27
4,Albania,2021,Raw cows' milk delivered to dairies,58.94


Lets convert the relevant variables from long format to wide format.

In [542]:
df_products_milk = df_products_milk .pivot_table(index=["geo","TIME_PERIOD"],columns="dairyprod",values="OBS_VALUE").reset_index()
df_products_milk.columns.name = None  # Resetting column index name
df_products_milk

,geo,TIME_PERIOD,Cream for direct consumption,Drinking milk,"Milk and cream powders, excluding skimmed milk powders",Raw cows' milk delivered to dairies,Raw cream delivered to dairies (in milk equivalent)
0,Albania,2017,0.21,8.93,0.00,57.36,0.0
1,Albania,2018,0.46,13.47,0.00,64.39,0.0
2,Albania,2019,0.23,10.15,0.00,56.82,0.0
3,Albania,2020,0.29,14.48,0.00,56.27,0.0
4,Albania,2021,0.29,12.19,0.00,58.94,0.0
...,...,...,...,...,...,...,...
362,Türkiye,2025,38.44,1639.76,39.86,NaN,NaN
363,United Kingdom,2016,289.60,6631.60,NaN,14542.00,0.0
364,United Kingdom,2017,307.20,6910.70,16.90,15144.70,0.0
365,United Kingdom,2018,284.90,6783.19,19.40,15188.10,0.0


In [543]:
df_products_milk.describe()

,TIME_PERIOD,Cream for direct consumption,Drinking milk,"Milk and cream powders, excluding skimmed milk powders",Raw cows' milk delivered to dairies,Raw cream delivered to dairies (in milk equivalent)
count,367.000000,313.000000,319.000000,278.000000,352.000000,222.000000
mean,2020.313351,91.258850,887.486865,24.264820,10959.964858,30.918423
std,2.907952,137.655781,1295.957456,40.950148,29688.736421,156.433609
min,2016.000000,0.210000,2.120000,0.000000,22.100000,0.000000
25%,2018.000000,14.360000,108.150000,0.000000,648.545000,0.000000
50%,2020.000000,31.930000,426.700000,0.945000,1884.215000,0.000000
75%,2023.000000,74.220000,726.825000,29.915000,7449.480000,0.000000
max,2025.000000,585.990000,6910.700000,184.100000,157382.630000,1164.730000


Lets analyze again if all features have non null values for Ireland, otherwise they will be dropped.

In [544]:
(df_products_milk[df_products_milk["geo"] == "Ireland"].fillna(0) != 0).any()

geo                                                        True
TIME_PERIOD                                                True
Cream for direct consumption                               True
Drinking milk                                              True
Milk and cream powders, excluding skimmed milk powders    False
Raw cows' milk delivered to dairies                        True
Raw cream delivered to dairies (in milk equivalent)       False
dtype: bool

Lets drop the features that don't add value to our analysis.

In [545]:
df_products_milk = df_products_milk.drop(columns=[
        "Milk and cream powders, excluding skimmed milk powders",
        "Raw cream delivered to dairies (in milk equivalent)" ])

df_products_milk = df_products_milk.rename(columns={
    "geo" : "country",
    "TIME_PERIOD": "year",
    "Cream for direct consumption": "cream_direct_consumption_1000t",
    "Drinking milk": "processed_drinking_milk_1000t",
    "Raw cows' milk delivered to dairies":"milk_delivered_to_dairies_1000t"
})

df_products_milk.head()

,country,year,cream_direct_consumption_1000t,processed_drinking_milk_1000t,milk_delivered_to_dairies_1000t
0,Albania,2017,0.21,8.93,57.36
1,Albania,2018,0.46,13.47,64.39
2,Albania,2019,0.23,10.15,56.82
3,Albania,2020,0.29,14.48,56.27
4,Albania,2021,0.29,12.19,58.94


In [546]:
df_products_milk

,country,year,cream_direct_consumption_1000t,processed_drinking_milk_1000t,milk_delivered_to_dairies_1000t
0,Albania,2017,0.21,8.93,57.36
1,Albania,2018,0.46,13.47,64.39
2,Albania,2019,0.23,10.15,56.82
3,Albania,2020,0.29,14.48,56.27
4,Albania,2021,0.29,12.19,58.94
...,...,...,...,...,...
362,Türkiye,2025,38.44,1639.76,NaN
363,United Kingdom,2016,289.60,6631.60,14542.00
364,United Kingdom,2017,307.20,6910.70,15144.70
365,United Kingdom,2018,284.90,6783.19,15188.10


In [547]:
df_products_milk.to_csv("data/processed/milk_products.csv", index=False)

## Butter Production Dataset

In [548]:
butter_prod =  pd.read_csv("data/raw/butter_production.csv")
butter_prod.head()

,DATAFLOW,LAST UPDATE,freq,dairyprod,milkitem,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TAG00038(1.0),07/10/25 23:00:00,Annual,"Butter, incl. dehydrated butter and ghee, and ...",Products obtained (1 000 t),Albania,2014,0.68,NaN,NaN
1,ESTAT:TAG00038(1.0),07/10/25 23:00:00,Annual,"Butter, incl. dehydrated butter and ghee, and ...",Products obtained (1 000 t),Albania,2015,0.94,p,NaN
2,ESTAT:TAG00038(1.0),07/10/25 23:00:00,Annual,"Butter, incl. dehydrated butter and ghee, and ...",Products obtained (1 000 t),Albania,2016,0.82,p,NaN
3,ESTAT:TAG00038(1.0),07/10/25 23:00:00,Annual,"Butter, incl. dehydrated butter and ghee, and ...",Products obtained (1 000 t),Albania,2017,0.88,NaN,NaN
4,ESTAT:TAG00038(1.0),07/10/25 23:00:00,Annual,"Butter, incl. dehydrated butter and ghee, and ...",Products obtained (1 000 t),Albania,2018,0.91,NaN,NaN


In [549]:
butter_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 406 entries, 0 to 405
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     406 non-null    object 
 1   LAST UPDATE  406 non-null    object 
 2   freq         406 non-null    object 
 3   dairyprod    406 non-null    object 
 4   milkitem     406 non-null    object 
 5   geo          406 non-null    object 
 6   TIME_PERIOD  406 non-null    int64  
 7   OBS_VALUE    371 non-null    float64
 8   OBS_FLAG     14 non-null     object 
 9   CONF_STATUS  35 non-null     object 
dtypes: float64(1), int64(1), object(8)
memory usage: 31.8+ KB


In [550]:
butter_prod["milkitem"].unique()

array(['Products obtained (1 000 t)'], dtype=object)

In [551]:
butter_prod["dairyprod"].unique()

array(['Butter, incl. dehydrated butter and ghee, and other fats and oils derived from milk; dairy spreads'],
      dtype=object)

In [552]:
butter_prod.describe()

,TIME_PERIOD,OBS_VALUE
count,406.000000,371.000000
mean,2018.480296,78.042345
std,3.410710,121.059653
min,2013.000000,0.000000
25%,2016.000000,4.365000
50%,2018.000000,27.980000
75%,2021.000000,93.550000
max,2024.000000,509.490000


In [553]:
butter_prod = butter_prod[["geo","TIME_PERIOD", "OBS_VALUE"]].reset_index(drop=True)
butter_prod = butter_prod.rename(columns={
    "geo" : "country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "butter_prod_1000t"
})
butter_prod.head()

,country,year,butter_prod_1000t
0,Albania,2014,0.68
1,Albania,2015,0.94
2,Albania,2016,0.82
3,Albania,2017,0.88
4,Albania,2018,0.91


In [554]:
butter_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 406 entries, 0 to 405
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   country            406 non-null    object 
 1   year               406 non-null    int64  
 2   butter_prod_1000t  371 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 9.6+ KB


In [555]:
butter_prod.to_csv("data/processed/butter_prod.csv", index=False)

In [556]:
## Cheese Production Dataset

In [557]:
cheese_prod =  pd.read_csv("data/raw/cheese_production.csv")
cheese_prod.head()

,DATAFLOW,LAST UPDATE,freq,milkitem,dairyprod,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TAG00040(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Cheese,Albania,2014,11.94,NaN,NaN
1,ESTAT:TAG00040(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Cheese,Albania,2015,13.50,p,NaN
2,ESTAT:TAG00040(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Cheese,Albania,2016,14.30,p,NaN
3,ESTAT:TAG00040(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Cheese,Albania,2017,14.70,NaN,NaN
4,ESTAT:TAG00040(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Cheese,Albania,2018,14.60,NaN,NaN


In [558]:
cheese_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     404 non-null    object 
 1   LAST UPDATE  404 non-null    object 
 2   freq         404 non-null    object 
 3   milkitem     404 non-null    object 
 4   dairyprod    404 non-null    object 
 5   geo          404 non-null    object 
 6   TIME_PERIOD  404 non-null    int64  
 7   OBS_VALUE    361 non-null    float64
 8   OBS_FLAG     11 non-null     object 
 9   CONF_STATUS  43 non-null     object 
dtypes: float64(1), int64(1), object(8)
memory usage: 31.7+ KB


In [559]:
cheese_prod.milkitem.unique()

array(['Products obtained (1 000 t)'], dtype=object)

In [560]:
cheese_prod.describe()

,TIME_PERIOD,OBS_VALUE
count,404.000000,361.000000
mean,2018.490099,371.701357
std,3.411759,559.915740
min,2013.000000,0.800000
25%,2016.000000,51.570000
50%,2018.000000,102.510000
75%,2021.000000,452.000000
max,2024.000000,2433.320000


In [561]:
cheese_prod = cheese_prod[["geo","TIME_PERIOD", "OBS_VALUE"]].reset_index(drop=True)
cheese_prod = cheese_prod.rename(columns={
    "geo":"country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "cheese_prod_1000t"
})
cheese_prod.head()

,country,year,cheese_prod_1000t
0,Albania,2014,11.94
1,Albania,2015,13.50
2,Albania,2016,14.30
3,Albania,2017,14.70
4,Albania,2018,14.60


In [562]:
cheese_prod.to_csv("data/processed/cheese_prod.csv", index=False)

## Milk Powder Production Dataset

In [563]:
milk_powder_prod =  pd.read_csv("data/raw/milk_powders_production.csv")
milk_powder_prod.head()

,DATAFLOW,LAST UPDATE,freq,milkitem,dairyprod,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TAG00039(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Powder products,Albania,2014,0.0,NaN,NaN
1,ESTAT:TAG00039(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Powder products,Albania,2015,0.0,p,NaN
2,ESTAT:TAG00039(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Powder products,Albania,2016,0.0,p,NaN
3,ESTAT:TAG00039(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Powder products,Albania,2017,0.0,NaN,NaN
4,ESTAT:TAG00039(1.0),07/10/25 23:00:00,Annual,Products obtained (1 000 t),Powder products,Albania,2018,0.0,NaN,NaN


In [564]:
milk_powder_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 407 entries, 0 to 406
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     407 non-null    object 
 1   LAST UPDATE  407 non-null    object 
 2   freq         407 non-null    object 
 3   milkitem     407 non-null    object 
 4   dairyprod    407 non-null    object 
 5   geo          407 non-null    object 
 6   TIME_PERIOD  407 non-null    int64  
 7   OBS_VALUE    299 non-null    float64
 8   OBS_FLAG     15 non-null     object 
 9   CONF_STATUS  108 non-null    object 
dtypes: float64(1), int64(1), object(8)
memory usage: 31.9+ KB


In [565]:
milk_powder_prod.milkitem.unique()

array(['Products obtained (1 000 t)'], dtype=object)

In [566]:
milk_powder_prod= milk_powder_prod[["geo","TIME_PERIOD", "OBS_VALUE"]].reset_index(drop=True)
milk_powder_prod = milk_powder_prod.rename(columns={
    "geo" : "country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "milk_powder_prod_1000t"
})
milk_powder_prod

,country,year,milk_powder_prod_1000t
0,Albania,2014,0.00
1,Albania,2015,0.00
2,Albania,2016,0.00
3,Albania,2017,0.00
4,Albania,2018,0.00
...,...,...,...
402,United Kingdom,2015,174.37
403,United Kingdom,2016,NaN
404,United Kingdom,2017,NaN
405,United Kingdom,2018,96.49


In [567]:
milk_powder_prod.describe()

,year,milk_powder_prod_1000t
count,407.000000,299.000000
mean,2018.454545,144.346622
std,3.427974,333.769658
min,2013.000000,0.000000
25%,2015.500000,0.000000
50%,2018.000000,33.990000
75%,2021.000000,134.250000
max,2024.000000,3055.660000


In [568]:
milk_powder_prod.to_csv("data/processed/milk_powder_prod.csv",index=False)

## Bovine Population Dataset

In [569]:
bovine_pop = pd.read_csv("data/raw/bovine_pop.csv")
bovine_pop.head()

,DATAFLOW,LAST UPDATE,freq,month,animals,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:APRO_MT_LSCATL$DEFAULTVIEW(1.0),18/03/26 23:00:00,Annual,May-June,Live bovine animals,Thousand heads (animals),Austria,2016,1932.53,NaN,NaN
1,ESTAT:APRO_MT_LSCATL$DEFAULTVIEW(1.0),18/03/26 23:00:00,Annual,May-June,Live bovine animals,Thousand heads (animals),Austria,2017,1938.48,NaN,NaN
2,ESTAT:APRO_MT_LSCATL$DEFAULTVIEW(1.0),18/03/26 23:00:00,Annual,May-June,Live bovine animals,Thousand heads (animals),Austria,2018,1906.82,NaN,NaN
3,ESTAT:APRO_MT_LSCATL$DEFAULTVIEW(1.0),18/03/26 23:00:00,Annual,May-June,Live bovine animals,Thousand heads (animals),Austria,2019,1873.31,NaN,NaN
4,ESTAT:APRO_MT_LSCATL$DEFAULTVIEW(1.0),18/03/26 23:00:00,Annual,May-June,Live bovine animals,Thousand heads (animals),Austria,2020,1844.34,NaN,NaN


In [570]:
bovine_pop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2747 entries, 0 to 2746
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     2747 non-null   object 
 1   LAST UPDATE  2747 non-null   object 
 2   freq         2747 non-null   object 
 3   month        2747 non-null   object 
 4   animals      2747 non-null   object 
 5   unit         2747 non-null   object 
 6   geo          2747 non-null   object 
 7   TIME_PERIOD  2747 non-null   int64  
 8   OBS_VALUE    2746 non-null   float64
 9   OBS_FLAG     190 non-null    object 
 10  CONF_STATUS  0 non-null      float64
dtypes: float64(2), int64(1), object(8)
memory usage: 236.2+ KB


In [571]:
bovine_pop.unit.unique()

array(['Thousand heads (animals)'], dtype=object)

In [572]:
bovine_pop = bovine_pop[["geo","TIME_PERIOD", "OBS_VALUE"]].reset_index(drop=True)
bovine_pop = bovine_pop.rename(columns={
    "geo" : "country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "bovine_pop_1000"
})
bovine_pop.head()

,country,year,bovine_pop_1000
0,Austria,2016,1932.53
1,Austria,2017,1938.48
2,Austria,2018,1906.82
3,Austria,2019,1873.31
4,Austria,2020,1844.34


In [573]:
bovine_pop.describe()

,year,bovine_pop_1000
count,2747.000000,2746.000000
mean,2020.382599,2714.060302
std,2.897803,8818.384933
min,2016.000000,0.000000
25%,2018.000000,99.775000
50%,2020.000000,422.240000
75%,2023.000000,1749.835000
max,2025.000000,89503.540000


In [574]:
bovine_pop.to_csv("data/processed/bovine_pop.csv",index=False)

In [575]:
## Dairy Cow Population Dataset

In [576]:
dairy_cow_pop = pd.read_csv("data/raw/dairy_cows_pop.csv")
dairy_cow_pop.head()

,DATAFLOW,LAST UPDATE,freq,month,animals,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TAG00014(1.0),18/03/26 23:00:00,Annual,November - December,Dairy cows,Thousand heads (animals),Albania,2014,357.78,NaN,NaN
1,ESTAT:TAG00014(1.0),18/03/26 23:00:00,Annual,November - December,Dairy cows,Thousand heads (animals),Albania,2015,357.52,NaN,NaN
2,ESTAT:TAG00014(1.0),18/03/26 23:00:00,Annual,November - December,Dairy cows,Thousand heads (animals),Albania,2016,353.05,NaN,NaN
3,ESTAT:TAG00014(1.0),18/03/26 23:00:00,Annual,November - December,Dairy cows,Thousand heads (animals),Albania,2017,346.41,NaN,NaN
4,ESTAT:TAG00014(1.0),18/03/26 23:00:00,Annual,November - December,Dairy cows,Thousand heads (animals),Albania,2018,339.94,NaN,NaN


In [577]:
dairy_cow_pop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 445 entries, 0 to 444
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     445 non-null    object 
 1   LAST UPDATE  445 non-null    object 
 2   freq         445 non-null    object 
 3   month        445 non-null    object 
 4   animals      445 non-null    object 
 5   unit         445 non-null    object 
 6   geo          445 non-null    object 
 7   TIME_PERIOD  445 non-null    int64  
 8   OBS_VALUE    445 non-null    float64
 9   OBS_FLAG     34 non-null     object 
 10  CONF_STATUS  0 non-null      float64
dtypes: float64(2), int64(1), object(8)
memory usage: 38.4+ KB


In [578]:
dairy_cow_pop.unit.unique()

array(['Thousand heads (animals)'], dtype=object)

In [579]:
dairy_cow_pop = dairy_cow_pop[["geo","TIME_PERIOD", "OBS_VALUE"]].reset_index(drop=True)
dairy_cow_pop = dairy_cow_pop.rename(columns={
    "geo" : "country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "dairy_cow_pop_1000"
})
dairy_cow_pop.head()

,country,year,dairy_cow_pop_1000
0,Albania,2014,357.78
1,Albania,2015,357.52
2,Albania,2016,353.05
3,Albania,2017,346.41
4,Albania,2018,339.94


In [580]:
dairy_cow_pop.describe()

,year,dairy_cow_pop_1000
count,445.000000,445.000000
mean,2019.433708,1341.475416
std,3.437765,3444.513588
min,2014.000000,5.510000
25%,2016.000000,114.900000
50%,2019.000000,282.910000
75%,2022.000000,1056.700000
max,2025.000000,21651.740000


In [581]:
dairy_cow_pop.to_csv("data/processed/dairy_cow_pop.csv",index=False)

## Selling Price for Farmers Dataset

In [582]:
raw_milk_price = pd.read_csv("data/raw/selling_price_milk.csv")
raw_milk_price.head()

,DATAFLOW,LAST UPDATE,freq,currency,prod_ani,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TAG00070(1.0),15/05/25 11:00:00,Annual,Euro,Raw cows' milk actual fat content - prices per...,Austria,2013,37.55,NaN,NaN
1,ESTAT:TAG00070(1.0),15/05/25 11:00:00,Annual,Euro,Raw cows' milk actual fat content - prices per...,Austria,2014,39.46,NaN,NaN
2,ESTAT:TAG00070(1.0),15/05/25 11:00:00,Annual,Euro,Raw cows' milk actual fat content - prices per...,Austria,2015,33.73,NaN,NaN
3,ESTAT:TAG00070(1.0),15/05/25 11:00:00,Annual,Euro,Raw cows' milk actual fat content - prices per...,Austria,2016,31.33,NaN,NaN
4,ESTAT:TAG00070(1.0),15/05/25 11:00:00,Annual,Euro,Raw cows' milk actual fat content - prices per...,Austria,2017,37.28,NaN,NaN


In [583]:
raw_milk_price.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290 entries, 0 to 289
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     290 non-null    object 
 1   LAST UPDATE  290 non-null    object 
 2   freq         290 non-null    object 
 3   currency     290 non-null    object 
 4   prod_ani     290 non-null    object 
 5   geo          290 non-null    object 
 6   TIME_PERIOD  290 non-null    int64  
 7   OBS_VALUE    280 non-null    float64
 8   OBS_FLAG     8 non-null      object 
 9   CONF_STATUS  7 non-null      object 
dtypes: float64(1), int64(1), object(8)
memory usage: 22.8+ KB


In [584]:
raw_milk_price.describe(include = "object")

,DATAFLOW,LAST UPDATE,freq,currency,prod_ani,geo,OBS_FLAG,CONF_STATUS
count,290,290,290,290,290,290,8,7
unique,1,1,1,1,1,26,2,1
top,ESTAT:TAG00070(1.0),15/05/25 11:00:00,Annual,Euro,Raw cows' milk actual fat content - prices per...,Austria,p,C
freq,290,290,290,290,290,12,5,7


In [585]:
raw_milk_price = raw_milk_price[["geo","TIME_PERIOD", "OBS_VALUE"]].reset_index(drop=True)
raw_milk_price = raw_milk_price.rename(columns={
    "geo" : "country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "farmers_selling_price_100kg"
})
raw_milk_price.head()

,country,year,farmers_selling_price_100kg
0,Austria,2013,37.55
1,Austria,2014,39.46
2,Austria,2015,33.73
3,Austria,2016,31.33
4,Austria,2017,37.28


In [586]:
raw_milk_price.to_csv("data/processed/raw_milk_price.csv",index=False)

## Economic Accounts

In [587]:
economic_account = pd.read_csv("data/raw/economic_account.csv")
economic_account.head()

,DATAFLOW,LAST UPDATE,freq,am_item,indic_agr,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:AACT_EAA01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Output of the agricultural 'industry',Production value at basic price,Million euro,Austria,2016,7000.76,NaN,NaN
1,ESTAT:AACT_EAA01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Output of the agricultural 'industry',Production value at basic price,Million euro,Austria,2017,7441.70,NaN,NaN
2,ESTAT:AACT_EAA01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Output of the agricultural 'industry',Production value at basic price,Million euro,Austria,2018,7500.74,NaN,NaN
3,ESTAT:AACT_EAA01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Output of the agricultural 'industry',Production value at basic price,Million euro,Austria,2019,7590.30,NaN,NaN
4,ESTAT:AACT_EAA01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Output of the agricultural 'industry',Production value at basic price,Million euro,Austria,2020,7720.12,NaN,NaN


In [588]:
economic_account.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4356 entries, 0 to 4355
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     4356 non-null   object 
 1   LAST UPDATE  4356 non-null   object 
 2   freq         4356 non-null   object 
 3   am_item      4356 non-null   object 
 4   indic_agr    4356 non-null   object 
 5   unit         4356 non-null   object 
 6   geo          4356 non-null   object 
 7   TIME_PERIOD  4356 non-null   int64  
 8   OBS_VALUE    4029 non-null   float64
 9   OBS_FLAG     759 non-null    object 
 10  CONF_STATUS  0 non-null      float64
dtypes: float64(2), int64(1), object(8)
memory usage: 374.5+ KB


In [589]:
economic_account.describe(include="object")

,DATAFLOW,LAST UPDATE,freq,am_item,indic_agr,unit,geo,OBS_FLAG
count,4356,4356,4356,4356,4356,4356,4356,759
unique,1,1,1,1,4,3,37,2
top,ESTAT:AACT_EAA01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Output of the agricultural 'industry',Production value at basic price,Million euro,Austria,e
freq,4356,4356,4356,4356,1089,1452,120,432


In [590]:
economic_account["indic_agr"].unique()

array(['Production value at basic price',
       'Production value at producer price', 'Subsidies on products',
       'Taxes on products'], dtype=object)

In [591]:
economic_account["unit"].unique()

array(['Million euro', 'Million units of national currency',
       'Million purchasing power standards (PPS)'], dtype=object)

In [592]:
economic_account = economic_account[economic_account["unit"] == "Million euro"][["geo","TIME_PERIOD","indic_agr","OBS_VALUE"]]
economic_account.head()

,geo,TIME_PERIOD,indic_agr,OBS_VALUE
0,Austria,2016,Production value at basic price,7000.76
1,Austria,2017,Production value at basic price,7441.70
2,Austria,2018,Production value at basic price,7500.74
3,Austria,2019,Production value at basic price,7590.30
4,Austria,2020,Production value at basic price,7720.12


In [593]:
economic_account = economic_account.pivot_table(index=["geo","TIME_PERIOD"],columns="indic_agr",values="OBS_VALUE").reset_index()
economic_account.columns.name = None  # Resetting column index name
economic_account.head()

,geo,TIME_PERIOD,Production value at basic price,Production value at producer price,Subsidies on products,Taxes on products
0,Austria,2016,7000.76,7002.95,5.80,7.99
1,Austria,2017,7441.70,7441.23,7.21,6.74
2,Austria,2018,7500.74,7505.15,3.74,8.14
3,Austria,2019,7590.30,7595.23,3.72,8.65
4,Austria,2020,7720.12,7721.69,6.73,8.30


In [594]:
economic_account.describe()

,TIME_PERIOD,Production value at basic price,Production value at producer price,Subsidies on products,Taxes on products
count,363.000000,363.000000,363.000000,343.000000,274.000000
mean,2020.443526,73764.369725,72961.067934,938.633994,110.777007
std,2.867995,149062.443203,147514.860424,1791.047693,210.356113
min,2016.000000,121.090000,121.090000,0.000000,0.000000
25%,2018.000000,2818.010000,2611.745000,40.555000,0.000000
50%,2020.000000,8864.290000,8785.030000,135.350000,0.000000
75%,2023.000000,43819.380000,43758.575000,547.865000,49.312500
max,2025.000000,581890.560000,575936.570000,7484.120000,730.330000


In [595]:
economic_account = economic_account.rename(columns={
    "geo": "country",
    "TIME_PERIOD": "year",
    "Production value at basic price": "agri_output_basic_price_million_eur",
    "Production value at producer price": "agri_output_producer_price_million_eur",
    "Subsidies on products": "agri_subsidies_million_eur",
    "Taxes on products": "agri_taxes_million_eur"
})
economic_account.head()

,country,year,agri_output_basic_price_million_eur,agri_output_producer_price_million_eur,agri_subsidies_million_eur,agri_taxes_million_eur
0,Austria,2016,7000.76,7002.95,5.80,7.99
1,Austria,2017,7441.70,7441.23,7.21,6.74
2,Austria,2018,7500.74,7505.15,3.74,8.14
3,Austria,2019,7590.30,7595.23,3.72,8.65
4,Austria,2020,7720.12,7721.69,6.73,8.30


In [596]:
economic_account.to_csv("data/processed/economic_account.csv",index=False)

## Labour Input (Annual Work Units) Dataset

In [597]:
labour_input_awu = pd.read_csv("data/raw/agriculture_labour_input.csv")
labour_input_awu.head()

,DATAFLOW,LAST UPDATE,freq,am_item,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:AACT_ALI01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Total agricultural labour input,Thousand annual working units (AWU),Austria,2016,121.07,NaN,NaN
1,ESTAT:AACT_ALI01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Total agricultural labour input,Thousand annual working units (AWU),Austria,2017,121.82,NaN,NaN
2,ESTAT:AACT_ALI01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Total agricultural labour input,Thousand annual working units (AWU),Austria,2018,121.30,NaN,NaN
3,ESTAT:AACT_ALI01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Total agricultural labour input,Thousand annual working units (AWU),Austria,2019,120.31,NaN,NaN
4,ESTAT:AACT_ALI01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Total agricultural labour input,Thousand annual working units (AWU),Austria,2020,121.61,NaN,NaN


In [598]:
labour_input_awu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1086 entries, 0 to 1085
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     1086 non-null   object 
 1   LAST UPDATE  1086 non-null   object 
 2   freq         1086 non-null   object 
 3   am_item      1086 non-null   object 
 4   unit         1086 non-null   object 
 5   geo          1086 non-null   object 
 6   TIME_PERIOD  1086 non-null   int64  
 7   OBS_VALUE    1086 non-null   float64
 8   OBS_FLAG     108 non-null    object 
 9   CONF_STATUS  0 non-null      float64
dtypes: float64(2), int64(1), object(7)
memory usage: 85.0+ KB


In [599]:
labour_input_awu.describe(include="object")

,DATAFLOW,LAST UPDATE,freq,am_item,unit,geo,OBS_FLAG
count,1086,1086,1086,1086,1086,1086,108
unique,1,1,1,3,1,37,1
top,ESTAT:AACT_ALI01$DEFAULTVIEW(1.0),13/05/26 11:00:00,Annual,Total agricultural labour input,Thousand annual working units (AWU),Austria,e
freq,1086,1086,1086,362,1086,30,108


In [600]:
labour_input_awu.am_item.unique()

array(['Total agricultural labour input',
       'Non salaried agricultural labour input',
       'Salaried agricultural labour input'], dtype=object)

In [601]:
labour_input_awu.unit.unique()

array(['Thousand annual working units (AWU)'], dtype=object)

In [602]:
labour_input_awu = labour_input_awu.pivot_table(index=["geo","TIME_PERIOD"],columns="am_item",values="OBS_VALUE").reset_index()
labour_input_awu.columns.name = None 
labour_input_awu  = labour_input_awu .rename(columns={
    "geo": "country",
    "TIME_PERIOD": "year",
    "Non salaried agricultural labour input": "non_salaried_labour_awu",
    "Salaried agricultural labour input": "salaried_labour_awu",
    "Total agricultural labour input": "total_labour_awu"
})
labour_input_awu.head()

,country,year,non_salaried_labour_awu,salaried_labour_awu,total_labour_awu
0,Austria,2016,102.69,18.39,121.07
1,Austria,2017,102.58,19.24,121.82
2,Austria,2018,101.52,19.78,121.30
3,Austria,2019,99.87,20.44,120.31
4,Austria,2020,101.64,19.97,121.61


In [603]:
labour_input_awu.to_csv("data/processed/labour_input_awu.csv",index=False)

In [604]:
## 